## LRModelQ3

In [1]:
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


PART_B_DIR = Path.cwd().parents[0]
DATA_FILE = PART_B_DIR / "data" / "yelp_clean.csv"
MODEL_DIR = PART_B_DIR / "models"

### Data Preparation

In [2]:
# --- Load Clean Data (Yelp 3-class sentiment) ---
df = pd.read_csv(DATA_FILE)

In [3]:
# --- Split into train/test (same split as Q2 so results stay comparable) ---
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['sentiment'],
    test_size=0.2, random_state=42, stratify=df['sentiment']
)

### Hyperparameter Search

In [4]:
# --- Base pipeline to tune (same two stages as Q2) ---
# The TF-IDF settings are deliberately left at their defaults here, because
# max_features, ngram_range, min_df and sublinear_tf are all searched below.
# strip_accents stays fixed since it is not a tuning decision.
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(strip_accents="unicode")),
    ('clf', LogisticRegression(max_iter=1000, n_jobs=-1, random_state=42))
])

In [5]:
# --- Hyperparameter search space ---
# Same TF-IDF search space across all 4 Q3 models for a fair comparison.
# 3 * 2 * 3 * 2 * 5 * 1 * 2 = 360 possible combinations, of which the search samples 10.
# penalty holds a single value, so it is fixed rather than searched. It is deprecated from
# scikit-learn 1.8 (see the warning below), but removing the key would change the order the
# sampler draws in, giving different best parameters and invalidating the Q4 results.
param_dist = {
    'tfidf__max_features': [10000, 20000, 30000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [1, 2, 3],
    'tfidf__sublinear_tf': [True, False],
    'clf__C': [0.1, 0.5, 1.0, 2.0, 5.0],
    'clf__penalty': ['l2'],
    'clf__class_weight': [None, 'balanced']
}

In [6]:
# --- Randomized search ---
# n_iter=10 samples 10 of the 360 combinations; cv=5 scores each one by 5-fold cross
# validation, so 50 fits in total.
# scoring='f1_macro' rather than accuracy: neutral is half the size of the other classes,
# so a model that simply avoids predicting neutral would still score well on accuracy.
search = RandomizedSearchCV(
    pipeline, param_dist,
    n_iter=10, cv=5,
    scoring='f1_macro',
    random_state=42, n_jobs=-1, verbose=0
)

search.fit(X_train, y_train)

C:\Users\yingx\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\yingx\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


### Search Results

In [7]:
# --- Report best hyperparameters and score ---
print("=== Q3: Best Hyperparameters ===")
print("=" * 45)
for param, value in search.best_params_.items():
    name = param.replace('clf__', '').replace('tfidf__', '')
    print(f"  {name:<20} : {value}")
print("=" * 45)
print("Best CV Macro F1:", search.best_score_)

=== Q3: Best Hyperparameters ===
  sublinear_tf         : True
  ngram_range          : (1, 1)
  min_df               : 2
  max_features         : 20000
  penalty              : l2
  class_weight         : balanced
  C                    : 0.5
Best CV Macro F1: 0.699672785723403


### Save Tuned Model

In [8]:
# --- Save tuned model for Q4 ---
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(search.best_estimator_, MODEL_DIR / 'lr_tuned.joblib')

['C:\\Users\\yingx\\Desktop\\TextAssignment\\Part_B\\models\\lr_tuned.joblib']